# Maximum Nash Welfare Extension with an LLM Proxy


This extension compares maximum Nash welfare allocations computed from true bundle utilities and from LLM-proxy preferences. Once the previously extracted LLM profiles are available, the allocation and evaluation sections run on CPU.

Methods:

1. **Oracle MNW**, using hidden true bundle utilities.
2. **Strict LLM-proxy MNW**, using normalized latent bundle scores.
3. **Weak LLM-proxy MNW**, using the PE-calibrated tiers locked in the earlier RSD experiment.

All allocations are evaluated using hidden true bundle utilities and strict true rankings. The notebook reports true Nash-welfare ratios, Pareto gaps, proportional-fairness gaps, and a course-specific equal-capacity proportional benchmark. Results are relative to each student's observed candidate-bundle domain plus the outside option.


## 1. Install packages


In [ ]:
!python -m pip install -q -U "transformers>=4.56,<5" "accelerate>=1.2" "bitsandbytes>=0.46.1" scikit-learn scipy sentencepiece gdown

In [ ]:
import ast
import json
import random
import re
import shutil
import time
from itertools import combinations
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy.optimize import Bounds, LinearConstraint, milp
from scipy.sparse import lil_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GroupKFold
from tqdm.auto import tqdm

SEED = 20260909
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
N_FOLDS = 5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


## 2. Load the synthetic course-allocation data


In [ ]:
from pathlib import Path
import json
import os
import shutil
import zipfile

import gdown
import numpy as np
import pandas as pd

# Use a local dataset ZIP, or opt in to a link-shared Drive download.
ZIP_PATH = Path("fixed_tolerance_course_data.zip")
FILE_ID = os.environ.get("COURSE_DATA_DRIVE_FILE_ID", "").strip()

if not ZIP_PATH.exists() or not zipfile.is_zipfile(ZIP_PATH):
    if not FILE_ID:
        raise FileNotFoundError(
            f"Place a valid {ZIP_PATH} in the working directory or set "
            "COURSE_DATA_DRIVE_FILE_ID to the ID of a link-shared dataset."
        )

    downloaded_path = gdown.download(
        id=FILE_ID,
        output=str(ZIP_PATH),
        quiet=False,
    )
    if downloaded_path is None:
        raise RuntimeError(
            "The Google Drive download failed. Check that anyone with the link "
            "can download the configured file."
        )

assert ZIP_PATH.exists(), f"Downloaded file not found: {ZIP_PATH}"
assert zipfile.is_zipfile(ZIP_PATH), (
    "The downloaded file is not a valid ZIP. Check the Google Drive "
    "sharing permission and FILE_ID."
)

print(
    f"Dataset ZIP: {ZIP_PATH} "
    f"({ZIP_PATH.stat().st_size / 1_000_000:.2f} MB)"
)

# Extract the dataset into a clean input directory.
INPUT_DIR = Path("fixed_tolerance_input")

if INPUT_DIR.exists():
    shutil.rmtree(INPUT_DIR)

INPUT_DIR.mkdir(parents=True)
shutil.unpack_archive(str(ZIP_PATH), INPUT_DIR)

# Locate the required files even if the ZIP contains a parent folder.
found = {
    path.name: path
    for path in INPUT_DIR.rglob("*")
    if path.is_file()
}

required = {
    "course_catalog.csv",
    "student_reports.csv",
    "student_ground_truth.json",
    "bundle_queries_with_truth.csv",
    "generation_metadata.json",
}

missing = required - set(found)

assert not missing, (
    f"Missing required files: {sorted(missing)}. "
    f"Files found: {sorted(found)}"
)

# Load the dataset.
catalog = pd.read_csv(found["course_catalog.csv"])
reports_df = pd.read_csv(found["student_reports.csv"])
queries = pd.read_csv(found["bundle_queries_with_truth.csv"])

metadata = json.loads(
    found["generation_metadata.json"].read_text(encoding="utf-8")
)

ground_truth = json.loads(
    found["student_ground_truth.json"].read_text(encoding="utf-8")
)

reports = (
    reports_df
    .set_index("student_id")["language_report"]
    .to_dict()
)

COURSE_IDS = sorted(
    catalog["course_id"].astype(int).tolist()
)

MAX_BUNDLE_SIZE = int(
    metadata["maximum_bundle_size"]
)

TRUE_COMMON_TOLERANCE = float(
    metadata["common_tolerance"]
)

# Convert serialized bundle lists back into Python lists.
for column in ["bundle_a", "bundle_b"]:
    queries[column] = queries[column].apply(json.loads)

# Student splits are generated later inside the mechanism-specific experiment.
student_ids = np.array(sorted(reports_df["student_id"].astype(int).unique()))
print("Students:", len(student_ids))
print("Queries:", len(queries))
print("True tolerance (used only to recover ordinal ground-truth tiers):", TRUE_COMMON_TOLERANCE)

display(catalog)
display(reports_df.sample(3, random_state=SEED))
display(queries.groupby("true_label").size().rename("count"))


## 3. Load the open-source LLM

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

assert torch.cuda.is_available(), "Select a GPU runtime in Colab."

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
).eval()

MODEL_REVISION = getattr(model.config, "_commit_hash", None)
print("Model:", MODEL_ID)
print("Revision:", MODEL_REVISION)
print("GPU:", torch.cuda.get_device_name(0))

## 4. Structured profile prompt and robust validation

In [ ]:
PUBLIC_CATALOG = catalog[
    ["course_id", "course_name", "field", "time_slot", "workload", "description"]
].to_dict("records")

PROFILE_PROMPT = '''
You extract a structured course-preference profile from a student's report.
Use only the report and the public course catalog. Never invent preferences.

For every course with clear evidence, assign exactly one attitude from:
STRONGLY_DISLIKE, DISLIKE, NEUTRAL, LIKE, STRONGLY_LIKE.
Omit courses without clear evidence; omitted courses are treated as NEUTRAL.

Extract complements only when two courses are especially valuable together.
Extract substitutes only when the student indicates overlap, redundancy, or a
desire not to take both. Represent a pair as "smaller_id|larger_id".

Return exactly one JSON object:
{
  "course_attitudes": [{"course_id": 3, "attitude": "LIKE"}],
  "complements": ["0|6"],
  "substitutes": ["2|7"]
}
Do not return Markdown, explanations, confidence scores, tolerance estimates,
or additional keys.
'''.strip()

ATTITUDE_VALUES = {
    "STRONGLY_DISLIKE": -2.0,
    "DISLIKE": -1.0,
    "NEUTRAL": 0.0,
    "LIKE": 1.0,
    "STRONGLY_LIKE": 2.0,
}
ATTITUDE_ALIASES = {
    "STRONG_DISLIKE": "STRONGLY_DISLIKE",
    "VERY_DISLIKE": "STRONGLY_DISLIKE",
    "STRONG_LIKE": "STRONGLY_LIKE",
    "VERY_LIKE": "STRONGLY_LIKE",
}


def extract_json_object(text):
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)
    start = text.find("{")
    end = text.rfind("}")
    if start < 0 or end < start:
        raise ValueError("No JSON object found.")
    return json.loads(text[start:end + 1])


def canonical_pair(value):
    if isinstance(value, str):
        pieces = value.split("|")
    elif isinstance(value, (list, tuple)) and len(value) == 2:
        pieces = value
    elif isinstance(value, dict):
        pieces = [value.get("course_id_1"), value.get("course_id_2")]
    else:
        raise ValueError(f"Invalid pair: {value}")
    a, b = sorted(map(int, pieces))
    if a == b or a not in COURSE_IDS or b not in COURSE_IDS:
        raise ValueError(f"Invalid course pair: {value}")
    return (a, b)


def validate_profile(parsed):
    if not isinstance(parsed, dict):
        raise ValueError("The parsed response is not an object.")

    course_scores = {course_id: 0.0 for course_id in COURSE_IDS}
    explicitly_scored = set()
    for item in parsed.get("course_attitudes", []):
        course_id = int(item["course_id"])
        attitude = str(item["attitude"]).strip().upper().replace(" ", "_")
        attitude = ATTITUDE_ALIASES.get(attitude, attitude)
        if course_id not in COURSE_IDS:
            raise ValueError(f"Unknown course ID: {course_id}")
        if attitude not in ATTITUDE_VALUES:
            raise ValueError(f"Unknown attitude: {attitude}")
        course_scores[course_id] = ATTITUDE_VALUES[attitude]
        explicitly_scored.add(course_id)

    complements = sorted(set(canonical_pair(x) for x in parsed.get("complements", [])))
    substitutes = sorted(set(canonical_pair(x) for x in parsed.get("substitutes", [])))
    overlap = set(complements) & set(substitutes)
    if overlap:
        # Ambiguous relationships are conservatively discarded.
        complements = [pair for pair in complements if pair not in overlap]
        substitutes = [pair for pair in substitutes if pair not in overlap]

    return {
        "course_scores": course_scores,
        "explicitly_scored_courses": sorted(explicitly_scored),
        "complements": complements,
        "substitutes": substitutes,
    }


@torch.inference_mode()
def generate_profile(report, max_new_tokens=420):
    payload = {"course_catalog": PUBLIC_CATALOG, "student_report": report}
    messages = [
        {"role": "system", "content": PROFILE_PROMPT},
        {"role": "user", "content": json.dumps(payload, ensure_ascii=False)},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated = output[0, inputs.input_ids.shape[1]:]
    raw = tokenizer.decode(generated, skip_special_tokens=True).strip()
    parsed = extract_json_object(raw)
    return validate_profile(parsed), raw

## 5. Smoke test

In [ ]:
smoke_student_id = int(reports_df.student_id.iloc[0])
smoke_profile, smoke_raw = generate_profile(reports[smoke_student_id])
print("Raw output:")
print(smoke_raw)
print("\nValidated profile:")
print(smoke_profile)

## 6. Extract one profile per student with checkpointing

In [ ]:
OUTPUT_DIR = Path("fixed_tolerance_experiment_results")
OUTPUT_DIR.mkdir(exist_ok=True)
CHECKPOINT = OUTPUT_DIR / "profile_extractions.jsonl"

completed = {}
if CHECKPOINT.exists():
    for line in CHECKPOINT.read_text(encoding="utf-8").splitlines():
        if line.strip():
            item = json.loads(line)
            completed[int(item["student_id"])] = item
print("Loaded checkpoint rows:", len(completed))

for student_id in tqdm(sorted(map(int, reports)), desc="Extracting profiles"):
    if student_id in completed and completed[student_id].get("status") == "ok":
        continue
    started = time.perf_counter()
    try:
        profile, raw = generate_profile(reports[student_id])
        item = {
            "student_id": student_id,
            "status": "ok",
            "profile": profile,
            "raw_output": raw,
            "error": None,
            "seconds": time.perf_counter() - started,
        }
    except Exception as exc:
        item = {
            "student_id": student_id,
            "status": "error",
            "profile": None,
            "raw_output": None,
            "error": repr(exc),
            "seconds": time.perf_counter() - started,
        }
    completed[student_id] = item
    with CHECKPOINT.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(item) + "\n")

status_df = pd.DataFrame([
    {key: value for key, value in item.items() if key not in {"profile", "raw_output"}}
    for item in completed.values()
]).sort_values("student_id")
display(status_df)
print(status_df.status.value_counts())

failed = status_df.loc[status_df.status != "ok", "student_id"].tolist()
assert not failed, f"Profile extraction failed for students: {failed}. Re-run this cell to retry them."

## 7. Convert profiles and bundle pairs into model features

In [ ]:
profiles = {student_id: item["profile"] for student_id, item in completed.items()}


def bundle_features(profile, bundle):
    bundle = tuple(sorted(map(int, bundle)))
    pairs = set(combinations(bundle, 2))
    score_sum = sum(float(profile["course_scores"].get(str(j), profile["course_scores"].get(j, 0.0))) for j in bundle)
    complement_count = sum(tuple(pair) in pairs for pair in profile["complements"])
    substitute_count = sum(tuple(pair) in pairs for pair in profile["substitutes"])
    explicit_count = sum(j in set(profile["explicitly_scored_courses"]) for j in bundle)
    return np.array([
        score_sum / MAX_BUNDLE_SIZE,
        complement_count / MAX_BUNDLE_SIZE,
        substitute_count / MAX_BUNDLE_SIZE,
        explicit_count / MAX_BUNDLE_SIZE,
    ], dtype=float)


feature_rows = []
for row in queries.itertuples(index=False):
    profile = profiles[int(row.student_id)]
    feature_difference = bundle_features(profile, row.bundle_a) - bundle_features(profile, row.bundle_b)
    feature_rows.append(feature_difference)

X = np.vstack(feature_rows)
FEATURE_NAMES = [
    "course_score_difference",
    "complement_difference",
    "substitute_difference",
    "explicit_coverage_difference",
]
for index, name in enumerate(FEATURE_NAMES):
    queries[name] = X[:, index]

display(queries.head())

The profile-extraction checkpoint uses the same filename as the earlier notebooks. If it is present in the Colab workspace, completed LLM profiles are reused. The allocation experiment below is CPU-only.


## 8. Preference model and locked weak thresholds


In [ ]:
N_REPEATED_SPLITS = 20
N_TRAIN_STUDENTS = 64
N_VALIDATION_STUDENTS = 32
N_TEST_STUDENTS = 32
N_VALIDATION_ORDERS = 20
N_TEST_ORDERS = 20
MARKET_CAPACITY_PER_COURSE = 5
LOGIT_C_GRID = [0.01, 0.1, 1.0, 10.0, 100.0]

# Thresholds are measured in within-student standardized predicted-score units.
# tau=0 is the strict-proxy baseline.
TAU_GRID = [0.0, 0.05, 0.10, 0.20, 0.30, 0.50, 0.75, 1.00]
ROBUST_CVAR_QUANTILE = 0.90
ROBUST_CVAR_WEIGHT = 0.50

market_capacities = {int(c): MARKET_CAPACITY_PER_COURSE for c in COURSE_IDS}


def strict_pair_target(rows):
    """Strict ground truth from raw utility; bundle tuple breaks exact equality."""
    targets = []
    for row in rows.itertuples(index=False):
        utility_a = float(row.true_utility_a)
        utility_b = float(row.true_utility_b)
        bundle_a = tuple(sorted(map(int, row.bundle_a)))
        bundle_b = tuple(sorted(map(int, row.bundle_b)))
        targets.append(int(
            utility_a > utility_b
            or (utility_a == utility_b and bundle_a < bundle_b)
        ))
    return np.asarray(targets, dtype=int)


def fit_strict_pairwise_model(rows, return_cv=False):
    """Fit a coherent linear bundle score using strict pair directions only."""
    X_local = rows[FEATURE_NAMES].to_numpy()
    y_local = strict_pair_target(rows)
    groups = rows.student_id.to_numpy()
    cv = GroupKFold(n_splits=min(N_FOLDS, rows.student_id.nunique()))
    c_rows = []
    for C in LOGIT_C_GRID:
        oof = np.full(len(rows), np.nan)
        for fit_idx, hold_idx in cv.split(X_local, y_local, groups):
            model = LogisticRegression(
                C=C, fit_intercept=False, solver="liblinear",
                random_state=SEED, max_iter=2_000,
            )
            model.fit(X_local[fit_idx], y_local[fit_idx])
            oof[hold_idx] = model.predict_proba(X_local[hold_idx])[:, 1]
        predicted = (oof >= 0.5).astype(int)
        c_rows.append({
            "C": C,
            "oof_direction_accuracy": accuracy_score(y_local, predicted),
            "oof_log_loss": float(-np.mean(
                y_local * np.log(np.clip(oof, 1e-12, 1))
                + (1 - y_local) * np.log(np.clip(1 - oof, 1e-12, 1))
            )),
        })
    cv_results = pd.DataFrame(c_rows).sort_values(
        ["oof_direction_accuracy", "oof_log_loss", "C"],
        ascending=[False, True, True],
    ).reset_index(drop=True)
    best_C = float(cv_results.loc[0, "C"])
    model = LogisticRegression(
        C=best_C, fit_intercept=False, solver="liblinear",
        random_state=SEED, max_iter=2_000,
    ).fit(X_local, y_local)
    return (model, best_C, cv_results) if return_cv else model


def heldout_direction_metrics(rows, model):
    y_local = strict_pair_target(rows)
    probabilities = model.predict_proba(rows[FEATURE_NAMES].to_numpy())[:, 1]
    predictions = (probabilities >= 0.5).astype(int)
    return {
        "direction_accuracy": float(accuracy_score(y_local, predictions)),
        "direction_log_loss": float(-np.mean(
            y_local * np.log(np.clip(probabilities, 1e-12, 1))
            + (1 - y_local) * np.log(np.clip(1 - probabilities, 1e-12, 1))
        )),
        "n_pairwise_queries": int(len(rows)),
    }


def build_candidate_market(rows, model):
    """Build standardized proxy scores and hidden true values on one market."""
    predicted, truth = {}, {}
    for student_id, student_rows in rows.groupby("student_id"):
        i = int(student_id)
        bundles = {
            tuple(sorted(map(int, b))) for b in student_rows.bundle_a
        } | {
            tuple(sorted(map(int, b))) for b in student_rows.bundle_b
        }
        raw = {
            b: float(np.dot(model.coef_[0], bundle_features(profiles[i], b)))
            for b in bundles
        }
        raw_values = np.array(list(raw.values()), dtype=float)
        scale = float(np.std(raw_values))
        if scale < 1e-8:
            scale = 1.0
        center = float(np.mean(raw_values))
        predicted[i] = {b: (value - center) / scale for b, value in raw.items()}

        observations = {b: [] for b in bundles}
        for row in student_rows.itertuples(index=False):
            a = tuple(sorted(map(int, row.bundle_a)))
            b = tuple(sorted(map(int, row.bundle_b)))
            observations[a].append(float(row.true_utility_a))
            observations[b].append(float(row.true_utility_b))
        truth[i] = {b: float(np.mean(values)) for b, values in observations.items()}

        # Outside option is always a separate, strictly worst tier.
        predicted[i][tuple()] = min(predicted[i].values()) - 10.0
        truth[i][tuple()] = min(truth[i].values()) - 2.0
    return predicted, truth


def strict_ranks(score_map):
    ordered = sorted(score_map, key=lambda b: (-score_map[b], b))
    return {bundle: rank for rank, bundle in enumerate(ordered)}


def rank_maps(score_maps):
    return {i: strict_ranks(scores) for i, scores in score_maps.items()}


def tier_map(score_map, tau):
    """Create transitive score-anchored tiers; the outside option stays worst."""
    outside = tuple()
    ordered = sorted((b for b in score_map if b), key=lambda b: (-score_map[b], b))
    tiers = {}
    if ordered:
        current_tier = 0
        anchor = score_map[ordered[0]]
        tiers[ordered[0]] = current_tier
        for bundle in ordered[1:]:
            if anchor - score_map[bundle] > tau + 1e-12:
                current_tier += 1
                anchor = score_map[bundle]
            tiers[bundle] = current_tier
        tiers[outside] = current_tier + 1
    else:
        tiers[outside] = 0
    return tiers


def tier_maps(score_maps, tau):
    return {i: tier_map(scores, tau) for i, scores in score_maps.items()}


def predicted_tie_rate(tiers):
    tied = total = 0
    for i in tiers:
        bundles = [b for b in tiers[i] if b]
        for a, b in combinations(bundles, 2):
            total += 1
            tied += int(tiers[i][a] == tiers[i][b])
    return float(tied / total) if total else 0.0


def fits(bundle, remaining):
    return all(remaining[c] > 0 for c in bundle)


def run_weak_rsd(decision_tiers, capacities, seed):
    """Weak RSD with a fixed report-independent lexicographic rule within a tier."""
    rng = np.random.default_rng(seed)
    order = list(map(int, rng.permutation(sorted(decision_tiers))))
    priority_position = {i: position for position, i in enumerate(order)}
    remaining = capacities.copy()
    allocation = {}
    for i in order:
        feasible = [b for b in decision_tiers[i] if fits(b, remaining)]
        best_tier = min(decision_tiers[i][b] for b in feasible)
        chosen = min(b for b in feasible if decision_tiers[i][b] == best_tier)
        allocation[i] = chosen
        for c in chosen:
            remaining[c] -= 1
    return allocation, priority_position


def proxy_pareto_completion(initial_allocation, proxy_tiers, capacities):
    """Proxy-Pareto completion: no proxy loss, maximum total tier improvement."""
    students = sorted(proxy_tiers)
    allowed = {
        i: [
            b for b in proxy_tiers[i]
            if proxy_tiers[i][b] <= proxy_tiers[i][initial_allocation[i]]
        ]
        for i in students
    }
    variables = [(i, b) for i in students for b in allowed[i]]
    index = {key: k for k, key in enumerate(variables)}
    rows, lower, upper = [], [], []

    for i in students:
        rows.append({index[(i, b)]: 1.0 for b in allowed[i]})
        lower.append(1.0); upper.append(1.0)
    for course in COURSE_IDS:
        rows.append({index[(i, b)]: 1.0 for i, b in variables if course in b})
        lower.append(-np.inf); upper.append(float(capacities[course]))

    matrix = lil_matrix((len(rows), len(variables)), dtype=float)
    for r, entries in enumerate(rows):
        for col, value in entries.items():
            matrix[r, col] = value

    tier_gain = np.array([
        proxy_tiers[i][initial_allocation[i]] - proxy_tiers[i][b]
        for i, b in variables
    ], dtype=float)
    # The tiny deterministic term resolves equal-gain optima without changing
    # the integer-valued primary objective.
    deterministic = np.array([
        (student_pos + 1) * 1e-7 + (bundle_pos + 1) * 1e-10
        for student_pos, i in enumerate(students)
        for bundle_pos, b in enumerate(allowed[i])
    ])
    objective = -tier_gain + deterministic
    result = milp(
        c=objective,
        integrality=np.ones(len(variables)),
        bounds=Bounds(np.zeros(len(variables)), np.ones(len(variables))),
        constraints=LinearConstraint(matrix.tocsr(), np.array(lower), np.array(upper)),
        options={"time_limit": 30.0},
    )
    if not result.success:
        raise RuntimeError(f"Pareto-completion MILP failed: {result.message}")
    completed = {i: b for (i, b), x in zip(variables, result.x) if x > 0.5}
    assert len(completed) == len(students)
    assert all(
        proxy_tiers[i][completed[i]] <= proxy_tiers[i][initial_allocation[i]]
        for i in students
    )
    return completed, int(round(tier_gain @ (result.x > 0.5))), result.message


def run_weak_rsd_with_completion(proxy_tiers, capacities, seed):
    initial, priority = run_weak_rsd(proxy_tiers, capacities, seed)
    completed, total_tier_gain, status = proxy_pareto_completion(
        initial, proxy_tiers, capacities
    )
    return completed, priority, initial, total_tier_gain, status


def run_strict_rsd(decision_ranks, capacities, seed):
    tiers = {i: dict(ranks) for i, ranks in decision_ranks.items()}
    return run_weak_rsd(tiers, capacities, seed)


def max_pareto_improvements(allocation, true_ranks, capacities):
    """Maximum strictly improved students subject to nobody becoming worse."""
    students = sorted(true_ranks)
    allowed = {
        i: [b for b in true_ranks[i] if true_ranks[i][b] <= true_ranks[i][allocation[i]]]
        for i in students
    }
    variables = [(i, b) for i in students for b in allowed[i]]
    index = {key: k for k, key in enumerate(variables)}
    rows, lower, upper = [], [], []
    for i in students:
        rows.append({index[(i, b)]: 1.0 for b in allowed[i]})
        lower.append(1.0); upper.append(1.0)
    for course in COURSE_IDS:
        rows.append({index[(i, b)]: 1.0 for i, b in variables if course in b})
        lower.append(-np.inf); upper.append(float(capacities[course]))
    matrix = lil_matrix((len(rows), len(variables)), dtype=float)
    for r, entries in enumerate(rows):
        for col, value in entries.items():
            matrix[r, col] = value
    objective = np.array([
        -float(true_ranks[i][b] < true_ranks[i][allocation[i]])
        for i, b in variables
    ])
    result = milp(
        c=objective,
        integrality=np.ones(len(variables)),
        bounds=Bounds(np.zeros(len(variables)), np.ones(len(variables))),
        constraints=LinearConstraint(matrix.tocsr(), np.array(lower), np.array(upper)),
        options={"time_limit": 30.0},
    )
    if not result.success:
        return np.nan, result.message
    chosen = {i: b for (i, b), x in zip(variables, result.x) if x > 0.5}
    count = sum(true_ranks[i][chosen[i]] < true_ranks[i][allocation[i]] for i in students)
    return int(count), result.message


def priority_blocking_metrics(allocation, true_ranks, capacities, priority_position):
    """Count strictly true-preferred bundles obtainable from empty/lower-priority seats."""
    holders = {c: [] for c in COURSE_IDS}
    for j, bundle in allocation.items():
        for c in bundle:
            holders[c].append(j)
    blocking = []
    for i in sorted(allocation):
        own = allocation[i]
        own_rank = true_ranks[i][own]
        for bundle, rank in true_ranks[i].items():
            if rank >= own_rank:
                continue
            obtainable = True
            for c in bundle:
                if c in own:
                    continue
                course_holders = holders[c]
                if len(course_holders) < capacities[c]:
                    continue
                if any(priority_position[i] < priority_position[j] for j in course_holders):
                    continue
                obtainable = False
                break
            if obtainable:
                blocking.append((i, bundle))
    denominator = sum(max(len(true_ranks[i]) - 1, 1) for i in true_ranks)
    return {
        "priority_blocking_bundles": len(blocking),
        "normalized_blocking_rate": float(len(blocking) / denominator),
        "priority_blocking_students": len({i for i, _ in blocking}),
        "zero_priority_blocking": int(len(blocking) == 0),
    }


def evaluate_allocation(allocation, true_ranks, capacities, priority_position):
    pe_count, solver_status = max_pareto_improvements(allocation, true_ranks, capacities)
    assigned_ranks = np.array([true_ranks[i][allocation[i]] for i in sorted(allocation)])
    metrics = {
        "pareto_improvable_students": pe_count,
        "pareto_violation_rate": float(pe_count / len(allocation)) if np.isfinite(pe_count) else np.nan,
        "pareto_efficient": int(pe_count == 0) if np.isfinite(pe_count) else np.nan,
        "pe_score": 1.0 - pe_count / len(allocation) if np.isfinite(pe_count) else np.nan,
        "mean_true_rank": float(assigned_ranks.mean()),
        "worst_true_rank": int(assigned_ranks.max()),
        "pe_solver_status": solver_status,
    }
    metrics.update(priority_blocking_metrics(
        allocation, true_ranks, capacities, priority_position
    ))
    return metrics


def compare_to_oracle(allocation, oracle_allocation, true_ranks):
    students = sorted(true_ranks)
    losses = np.array([
        true_ranks[i][allocation[i]] - true_ranks[i][oracle_allocation[i]]
        for i in students
    ])
    return {
        "allocation_agreement_rate": float(np.mean([
            allocation[i] == oracle_allocation[i] for i in students
        ])),
        "mean_true_rank_loss_vs_oracle": float(losses.mean()),
        "median_true_rank_loss_vs_oracle": float(np.median(losses)),
        "share_worse_than_oracle": float(np.mean(losses > 0)),
        "share_better_than_oracle": float(np.mean(losses < 0)),
        "worst_student_rank_loss": int(losses.max()),
    }


def cvar(values, quantile=ROBUST_CVAR_QUANTILE):
    values = np.asarray(values, dtype=float)
    cutoff = np.quantile(values, quantile)
    tail = values[values >= cutoff - 1e-12]
    return float(tail.mean())


def robust_risk(values):
    values = np.asarray(values, dtype=float)
    return float(values.mean() + ROBUST_CVAR_WEIGHT * cvar(values))


In [ ]:
from scipy.optimize import linprog

EPSILON_UTILITY = 1e-4

# Locked by validation in the earlier RSD weak-preference experiment. Reusing
# these values tests whether the learned indifference representation transfers
# to MNW without tuning on MNW test outcomes.
PE_TAU_BY_SPLIT = {
    0: 1.00, 1: 0.50, 2: 0.75, 3: 1.00, 4: 0.75,
    5: 1.00, 6: 0.50, 7: 1.00, 8: 1.00, 9: 0.30,
    10: 0.50, 11: 0.50, 12: 0.50, 13: 1.00, 14: 0.50,
    15: 1.00, 16: 1.00, 17: 1.00, 18: 1.00, 19: 0.50,
}


## 9. MNW, Pareto-gap, and proportionality evaluators


In [ ]:
def positive_values_from_scores(score_map, epsilon=EPSILON_UTILITY):
    """Candidate-menu min-max values in [epsilon, 1]; outside is epsilon."""
    nonempty_values = [float(value) for bundle, value in score_map.items() if bundle]
    if not nonempty_values:
        return {tuple(): float(epsilon)}
    floor = min(nonempty_values)
    top = max(nonempty_values)
    denominator = top - floor
    if denominator <= 1e-12:
        values = {bundle: 1.0 for bundle in score_map if bundle}
    else:
        values = {
            bundle: float(epsilon + (1.0 - epsilon) * (float(value) - floor) / denominator)
            for bundle, value in score_map.items() if bundle
        }
    values[tuple()] = float(epsilon)
    return values


def positive_values_from_tiers(tiers, epsilon=EPSILON_UTILITY):
    """Map best tier to one and the outside/worst tier to epsilon."""
    worst = max(tiers.values())
    if worst <= 0:
        return {bundle: 1.0 for bundle in tiers}
    return {
        bundle: float(epsilon + (1.0 - epsilon) * (worst - tier) / worst)
        for bundle, tier in tiers.items()
    }


def value_maps_from_scores(score_maps):
    return {i: positive_values_from_scores(scores) for i, scores in score_maps.items()}


def value_maps_from_tiers(tier_maps_by_student):
    return {i: positive_values_from_tiers(tiers) for i, tiers in tier_maps_by_student.items()}


def solve_mnw(value_maps, capacities):
    """Exact candidate-domain MNW via a binary linear program in log values."""
    students = sorted(value_maps)
    variables = [(i, b) for i in students for b in value_maps[i]]
    index = {key: k for k, key in enumerate(variables)}
    rows, lower, upper = [], [], []
    for i in students:
        rows.append({index[(i, b)]: 1.0 for b in value_maps[i]})
        lower.append(1.0); upper.append(1.0)
    for course in COURSE_IDS:
        rows.append({index[(i, b)]: 1.0 for i, b in variables if course in b})
        lower.append(-np.inf); upper.append(float(capacities[course]))
    matrix = lil_matrix((len(rows), len(variables)), dtype=float)
    for row_index, entries in enumerate(rows):
        for column, value in entries.items():
            matrix[row_index, column] = value
    objective = -np.array([
        np.log(max(float(value_maps[i][b]), EPSILON_UTILITY))
        for i, b in variables
    ])
    result = milp(
        c=objective,
        integrality=np.ones(len(variables)),
        bounds=Bounds(np.zeros(len(variables)), np.ones(len(variables))),
        constraints=LinearConstraint(matrix.tocsr(), np.array(lower), np.array(upper)),
        options={"time_limit": 60.0, "mip_rel_gap": 1e-9},
    )
    if not result.success:
        raise RuntimeError(f"MNW MILP failed: {result.message}")
    allocation = {i: b for (i, b), x in zip(variables, result.x) if x > 0.5}
    assert len(allocation) == len(students)
    log_nsw = float(np.mean([
        np.log(max(value_maps[i][allocation[i]], EPSILON_UTILITY)) for i in students
    ]))
    return allocation, log_nsw, result.message


def max_pareto_rank_gain(allocation, true_ranks, capacities):
    """Maximum total ordinal-rank gain subject to no student becoming worse."""
    students = sorted(true_ranks)
    allowed = {
        i: [b for b in true_ranks[i] if true_ranks[i][b] <= true_ranks[i][allocation[i]]]
        for i in students
    }
    variables = [(i, b) for i in students for b in allowed[i]]
    index = {key: k for k, key in enumerate(variables)}
    rows, lower, upper = [], [], []
    for i in students:
        rows.append({index[(i, b)]: 1.0 for b in allowed[i]})
        lower.append(1.0); upper.append(1.0)
    for course in COURSE_IDS:
        rows.append({index[(i, b)]: 1.0 for i, b in variables if course in b})
        lower.append(-np.inf); upper.append(float(capacities[course]))
    matrix = lil_matrix((len(rows), len(variables)), dtype=float)
    for row_index, entries in enumerate(rows):
        for column, value in entries.items():
            matrix[row_index, column] = value
    gains = np.array([
        true_ranks[i][allocation[i]] - true_ranks[i][b] for i, b in variables
    ], dtype=float)
    result = milp(
        c=-gains,
        integrality=np.ones(len(variables)),
        bounds=Bounds(np.zeros(len(variables)), np.ones(len(variables))),
        constraints=LinearConstraint(matrix.tocsr(), np.array(lower), np.array(upper)),
        options={"time_limit": 30.0},
    )
    if not result.success:
        raise RuntimeError(f"Pareto-rank-gap MILP failed: {result.message}")
    total_gain = float(gains @ (result.x > 0.5))
    denominator = float(sum(true_ranks[i][allocation[i]] for i in students))
    normalized_gap = total_gain / denominator if denominator > 0 else 0.0
    return total_gain, float(normalized_gap), result.message


def proportional_fairness_gap(allocation, true_values, capacities):
    """Maximum mean aggregate proportional utility improvement over allocation."""
    students = sorted(true_values)
    variables = [(i, b) for i in students for b in true_values[i]]
    index = {key: k for k, key in enumerate(variables)}
    rows, lower, upper = [], [], []
    for i in students:
        rows.append({index[(i, b)]: 1.0 for b in true_values[i]})
        lower.append(1.0); upper.append(1.0)
    for course in COURSE_IDS:
        rows.append({index[(i, b)]: 1.0 for i, b in variables if course in b})
        lower.append(-np.inf); upper.append(float(capacities[course]))
    matrix = lil_matrix((len(rows), len(variables)), dtype=float)
    for row_index, entries in enumerate(rows):
        for column, value in entries.items():
            matrix[row_index, column] = value
    proportional_changes = np.array([
        (true_values[i][b] - true_values[i][allocation[i]])
        / max(true_values[i][allocation[i]], EPSILON_UTILITY)
        for i, b in variables
    ], dtype=float)
    result = milp(
        c=-proportional_changes,
        integrality=np.ones(len(variables)),
        bounds=Bounds(np.zeros(len(variables)), np.ones(len(variables))),
        constraints=LinearConstraint(matrix.tocsr(), np.array(lower), np.array(upper)),
        options={"time_limit": 30.0},
    )
    if not result.success:
        raise RuntimeError(f"PF-gap MILP failed: {result.message}")
    gap = float(proportional_changes @ (result.x > 0.5)) / len(students)
    return max(gap, 0.0), result.message


def equal_capacity_share_benchmarks(true_values, capacities):
    """Individual fractional benchmark using 1/n of every course capacity."""
    students = sorted(true_values)
    n = len(students)
    benchmarks = {}
    for i in students:
        bundles = list(true_values[i])
        objective = -np.array([true_values[i][b] for b in bundles], dtype=float)
        A_eq = np.ones((1, len(bundles)), dtype=float)
        b_eq = np.array([1.0])
        A_ub = np.array([
            [float(course in bundle) for bundle in bundles] for course in COURSE_IDS
        ], dtype=float)
        b_ub = np.array([float(capacities[course]) / n for course in COURSE_IDS])
        result = linprog(
            c=objective, A_ub=A_ub, b_ub=b_ub,
            A_eq=A_eq, b_eq=b_eq,
            bounds=[(0.0, 1.0)] * len(bundles), method="highs",
        )
        if not result.success:
            raise RuntimeError(f"Equal-capacity benchmark LP failed for student {i}: {result.message}")
        benchmarks[i] = float(-result.fun)
    return benchmarks


def evaluate_mnw_allocation(
    allocation, true_values, true_ranks, capacities,
    oracle_log_nsw, equal_share_benchmarks,
):
    students = sorted(true_values)
    assigned_values = np.array([true_values[i][allocation[i]] for i in students])
    assigned_ranks = np.array([true_ranks[i][allocation[i]] for i in students])
    log_nsw = float(np.mean(np.log(np.maximum(assigned_values, EPSILON_UTILITY))))
    nsw_ratio = float(np.exp(log_nsw - oracle_log_nsw))
    pe_count, pe_status = max_pareto_improvements(allocation, true_ranks, capacities)
    rank_gain, rank_gap, rank_status = max_pareto_rank_gain(allocation, true_ranks, capacities)
    pf_gap, pf_status = proportional_fairness_gap(allocation, true_values, capacities)
    prop_ratios = np.array([
        true_values[i][allocation[i]] / max(equal_share_benchmarks[i], EPSILON_UTILITY)
        for i in students
    ])
    return {
        "true_log_nsw": log_nsw,
        "true_nsw_ratio_to_oracle": min(nsw_ratio, 1.0 + 1e-8),
        "pareto_improvable_students": int(pe_count),
        "pareto_efficient": int(pe_count == 0),
        "total_pareto_rank_gain": rank_gain,
        "normalized_ordinal_pareto_gap": rank_gap,
        "proportional_fairness_gap": pf_gap,
        "proportionally_fair": int(pf_gap <= 1e-8),
        "mean_true_value": float(assigned_values.mean()),
        "minimum_true_value": float(assigned_values.min()),
        "p10_true_value": float(np.quantile(assigned_values, 0.10)),
        "mean_true_rank": float(assigned_ranks.mean()),
        "worst_true_rank": int(assigned_ranks.max()),
        "outside_option_rate": float(np.mean([allocation[i] == tuple() for i in students])),
        "mean_equal_share_ratio": float(prop_ratios.mean()),
        "minimum_equal_share_ratio": float(prop_ratios.min()),
        "p10_equal_share_ratio": float(np.quantile(prop_ratios, 0.10)),
        "share_meeting_equal_share_benchmark": float(np.mean(prop_ratios >= 1.0 - 1e-8)),
        "pe_solver_status": pe_status,
        "rank_gap_solver_status": rank_status,
        "pf_solver_status": pf_status,
    }


## 10. Repeated locked-test MNW experiment


In [ ]:
all_ids = np.array(sorted(queries.student_id.unique()))
assert len(all_ids) == N_TRAIN_STUDENTS + N_VALIDATION_STUDENTS + N_TEST_STUDENTS
split_rng = np.random.default_rng(SEED + 2_400)

manifest_rows = []
run_rows = []
pairwise_rows = []

for split_id in tqdm(range(N_REPEATED_SPLITS), desc="MNW repeated splits"):
    shuffled = split_rng.permutation(all_ids)
    train_ids = set(map(int, shuffled[:N_TRAIN_STUDENTS]))
    validation_ids = set(map(int, shuffled[N_TRAIN_STUDENTS:N_TRAIN_STUDENTS + N_VALIDATION_STUDENTS]))
    test_ids = set(map(int, shuffled[-N_TEST_STUDENTS:]))
    for role, ids in [("train", train_ids), ("validation", validation_ids), ("test", test_ids)]:
        manifest_rows.extend({"split_id": split_id, "student_id": i, "role": role} for i in ids)

    train = queries[queries.student_id.isin(train_ids)].copy()
    test = queries[queries.student_id.isin(test_ids)].copy()
    fitted_model, best_C, _ = fit_strict_pairwise_model(train, return_cv=True)
    predicted_scores, true_scores = build_candidate_market(test, fitted_model)
    true_values = value_maps_from_scores(true_scores)
    strict_proxy_values = value_maps_from_scores(predicted_scores)
    true_ranks = rank_maps(true_scores)
    pe_tau = float(PE_TAU_BY_SPLIT[split_id])
    weak_tiers = tier_maps(predicted_scores, pe_tau)
    weak_proxy_values = value_maps_from_tiers(weak_tiers)

    oracle_allocation, oracle_log_nsw, oracle_status = solve_mnw(true_values, market_capacities)
    strict_allocation, _, strict_status = solve_mnw(strict_proxy_values, market_capacities)
    weak_allocation, _, weak_status = solve_mnw(weak_proxy_values, market_capacities)
    equal_share = equal_capacity_share_benchmarks(true_values, market_capacities)

    methods = {
        "oracle_mnw": (oracle_allocation, oracle_status),
        "llm_proxy_strict_mnw": (strict_allocation, strict_status),
        "llm_proxy_weak_mnw": (weak_allocation, weak_status),
    }
    for method, (allocation, solver_status) in methods.items():
        metrics = evaluate_mnw_allocation(
            allocation, true_values, true_ranks, market_capacities,
            oracle_log_nsw, equal_share,
        )
        run_rows.append({
            "split_id": split_id, "method": method,
            "best_logit_C": best_C, "locked_pe_tau": pe_tau,
            "mnw_solver_status": solver_status,
            **metrics,
        })
        if method != "oracle_mnw":
            students = sorted(true_ranks)
            losses = np.array([
                true_ranks[i][allocation[i]] - true_ranks[i][oracle_allocation[i]]
                for i in students
            ])
            pairwise_rows.append({
                "split_id": split_id, "method": method,
                "allocation_agreement_rate": float(np.mean([
                    allocation[i] == oracle_allocation[i] for i in students
                ])),
                "mean_true_rank_loss_vs_oracle": float(losses.mean()),
                "share_worse_than_oracle": float(np.mean(losses > 0)),
                "share_better_than_oracle": float(np.mean(losses < 0)),
                "worst_student_rank_loss": int(losses.max()),
            })

split_manifest = pd.DataFrame(manifest_rows)
run_results = pd.DataFrame(run_rows)
paired_comparisons = pd.DataFrame(pairwise_rows)

summary = run_results.groupby("method", as_index=False).agg(
    true_nsw_ratio_to_oracle=("true_nsw_ratio_to_oracle", "mean"),
    pe_rate=("pareto_efficient", "mean"),
    mean_pareto_improvable_students=("pareto_improvable_students", "mean"),
    mean_ordinal_pareto_gap=("normalized_ordinal_pareto_gap", "mean"),
    proportional_fair_rate=("proportionally_fair", "mean"),
    mean_pf_gap=("proportional_fairness_gap", "mean"),
    median_pf_gap=("proportional_fairness_gap", "median"),
    p90_pf_gap=("proportional_fairness_gap", lambda x: float(np.quantile(x, 0.90))),
    worst_pf_gap=("proportional_fairness_gap", "max"),
    mean_true_value=("mean_true_value", "mean"),
    mean_minimum_true_value=("minimum_true_value", "mean"),
    mean_p10_true_value=("p10_true_value", "mean"),
    mean_true_rank=("mean_true_rank", "mean"),
    mean_minimum_equal_share_ratio=("minimum_equal_share_ratio", "mean"),
    mean_share_meeting_equal_share=("share_meeting_equal_share_benchmark", "mean"),
)
comparison_summary = paired_comparisons.groupby("method", as_index=False).agg(
    allocation_agreement_rate=("allocation_agreement_rate", "mean"),
    mean_true_rank_loss_vs_oracle=("mean_true_rank_loss_vs_oracle", "mean"),
    share_worse_than_oracle=("share_worse_than_oracle", "mean"),
    share_better_than_oracle=("share_better_than_oracle", "mean"),
    worst_student_rank_loss=("worst_student_rank_loss", "max"),
)

display(summary)
display(comparison_summary)

oracle_rows = run_results[run_results.method.eq("oracle_mnw")]
assert int((oracle_rows.pareto_efficient != 1).sum()) == 0, "Oracle MNW must be candidate-domain PE."
assert np.all(oracle_rows.true_nsw_ratio_to_oracle.between(1 - 1e-7, 1 + 1e-7))
assert np.all(run_results.true_nsw_ratio_to_oracle <= 1 + 1e-6)
print("Oracle MNW sanity checks passed.")


## 11. Visualize and export


In [ ]:
FIGURE_DIR = Path("mnw_llm_proxy_figures")
FIGURE_DIR.mkdir(exist_ok=True)
method_order = ["oracle_mnw", "llm_proxy_strict_mnw", "llm_proxy_weak_mnw"]
labels = ["Oracle MNW", "Strict proxy", "Weak proxy"]
colors = ["#64748b", "#2563eb", "#7c3aed"]
plot_summary = summary.set_index("method").reindex(method_order)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].bar(labels, plot_summary.true_nsw_ratio_to_oracle, color=colors)
axes[0].set(title="True Nash welfare", ylabel="Geometric-mean ratio to Oracle", ylim=(0, 1.05))
axes[1].bar(labels, plot_summary.mean_ordinal_pareto_gap, color=colors)
axes[1].set(title="Distance to PE frontier", ylabel="Mean normalized ordinal PE gap")
axes[2].bar(labels, plot_summary.mean_pf_gap, color=colors)
axes[2].set(title="Proportional-fairness gap", ylabel="Mean PF gap")
for ax in axes:
    ax.grid(axis="y", alpha=0.25); ax.tick_params(axis="x", rotation=15)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "mnw_efficiency_and_pf.png", dpi=180)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(labels, plot_summary.mean_minimum_true_value, color=colors)
axes[0].set(title="Worst-off welfare", ylabel="Mean minimum normalized true value")
axes[1].bar(labels, plot_summary.mean_share_meeting_equal_share, color=colors)
axes[1].set(title="Equal-capacity proportional benchmark", ylabel="Mean student share meeting benchmark", ylim=(0, 1.05))
for ax in axes:
    ax.grid(axis="y", alpha=0.25); ax.tick_params(axis="x", rotation=15)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "mnw_distributional_fairness.png", dpi=180)
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
data = [
    run_results.loc[run_results.method.eq(method), "proportional_fairness_gap"].to_numpy()
    for method in method_order
]
ax.boxplot(data, labels=labels, showmeans=True)
ax.set(title="PF-gap distribution across locked-test markets", ylabel="Proportional-fairness gap")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "mnw_pf_gap_distribution.png", dpi=180)
plt.show()

OUTPUT_DIR = Path("mnw_llm_proxy_results")
OUTPUT_DIR.mkdir(exist_ok=True)
split_manifest.to_csv(OUTPUT_DIR / "split_manifest.csv", index=False)
run_results.to_csv(OUTPUT_DIR / "mnw_runs.csv", index=False)
paired_comparisons.to_csv(OUTPUT_DIR / "proxy_oracle_comparisons.csv", index=False)
summary.to_csv(OUTPUT_DIR / "summary.csv", index=False)
comparison_summary.to_csv(OUTPUT_DIR / "comparison_summary.csv", index=False)
for path in FIGURE_DIR.glob("*.png"):
    shutil.copy2(path, OUTPUT_DIR / path.name)

metadata = {
    "seed": SEED,
    "mechanism": "candidate-domain maximum Nash welfare",
    "methods": ["oracle true-utility MNW", "strict LLM-proxy MNW", "weak tier-based LLM-proxy MNW"],
    "true_utility_normalization": "within-student candidate-menu min-max to [epsilon, 1], with outside fixed at epsilon",
    "proxy_utility_normalization": "strict latent score uses the same candidate-menu min-max map; weak proxy uses tier position; outside is epsilon",
    "epsilon_utility": EPSILON_UTILITY,
    "weak_threshold_source": "locked split-specific PE thresholds from the earlier RSD validation experiment",
    "domain": "observed candidate bundles plus outside option",
    "proportionality_warning": "PF gap is evaluated on the discrete feasible domain; the equal-capacity-share ratio is a course-specific benchmark, not the standard additive-goods PROP1 theorem.",
    "n_repeated_splits": N_REPEATED_SPLITS,
    "split_sizes": {"train": N_TRAIN_STUDENTS, "validation": N_VALIDATION_STUDENTS, "locked_test": N_TEST_STUDENTS},
    "capacity_per_course": MARKET_CAPACITY_PER_COURSE,
    "locked_pe_thresholds_by_split": PE_TAU_BY_SPLIT,
}
(OUTPUT_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
archive = Path(shutil.make_archive(str(OUTPUT_DIR), "zip", OUTPUT_DIR))
print("Saved:", archive.resolve())
from google.colab import files
files.download(str(archive))
